# 🤖 RAG PDF Chatbot — Hybrid Retrieval + Evaluation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LaibaSarwar712/rag-pdf-chatbot/blob/main/RAG_PDF_Chatbot.ipynb)

**Goal:** Kisi bhi PDF se sawal poochna aur jawab page citation ke saath lena, bina hallucination ke (jawab sirf PDF ke andar se).

**Is project mein advanced cheezein:**
- Chunking with overlap
- Multilingual dense embeddings (`multilingual-e5-small`) + FAISS
- Sparse retrieval (BM25)
- **Hybrid retrieval** (Reciprocal Rank Fusion)
- Open-source LLM (`Qwen2.5-1.5B-Instruct`) se grounded answer generation
- **Retrieval evaluation** (Hit@k, MRR) jo dense vs BM25 vs hybrid compare karta hai
- Gradio demo

> ⚠️ **Zaroori:** Runtime → Change runtime type → **T4 GPU** select karein, phir **Run all**.

## 1. Setup

In [ ]:
!pip install -q pypdf sentence-transformers faiss-cpu rank_bm25 gradio transformers accelerate

import re, os, random, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

warnings.filterwarnings("ignore")
random.seed(42); np.random.seed(42)
os.makedirs("images", exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cpu":
    print("⚠️ GPU off hai. Runtime -> Change runtime type -> T4 GPU select karein, warna bohat slow chalega.")

## 2. PDF load karna
Default mein famous paper **"Attention Is All You Need"** download hota hai taake notebook seedha chal jaye.

Apni PDF use karni ho to **Option B** ke cell ko uncomment karein aur Option A ko comment kar dein.

In [ ]:
# Option A (default): demo paper download
!wget -q -O paper.pdf https://arxiv.org/pdf/1706.03762
PDF_PATH = "paper.pdf"

In [ ]:
# Option B: apni PDF upload karein
# from google.colab import files
# uploaded = files.upload()
# PDF_PATH = list(uploaded.keys())[0]

In [ ]:
from pypdf import PdfReader

def load_pdf(path):
    reader = PdfReader(path)
    pages = []
    for i, p in enumerate(reader.pages):
        t = re.sub(r"\s+", " ", p.extract_text() or "").strip()
        if t:
            pages.append({"page": i + 1, "text": t})
    return pages

pages = load_pdf(PDF_PATH)
print("Pages with text:", len(pages))
print(pages[0]["text"][:400])

## 3. Chunking
Lambi text ko chhote tukron (chunks) mein todte hain, **overlap** ke saath, taake koi jawab do chunks ke beech mein kat na jaye.

In [ ]:
def chunk_pages(pages, size=180, overlap=40):
    chunks, step = [], size - overlap
    for pg in pages:
        words = pg["text"].split()
        for s in range(0, max(len(words), 1), step):
            piece = words[s:s + size]
            if len(piece) >= 30:
                chunks.append({"id": len(chunks), "page": pg["page"], "text": " ".join(piece)})
            if s + size >= len(words):
                break
    return chunks

chunks = chunk_pages(pages)
print("Total chunks:", len(chunks))
print("Average words per chunk:", int(np.mean([len(c["text"].split()) for c in chunks])))

## 4. Retrieval: Dense + BM25 + Hybrid

- **Dense:** meaning ke hisab se dhoondta hai (embeddings + FAISS). Multilingual hai, is liye Urdu sawal bhi samajh sakta hai.
- **BM25:** exact keywords se dhoondta hai.
- **Hybrid:** dono ki rankings ko **Reciprocal Rank Fusion (RRF)** se milata hai.

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

embedder = SentenceTransformer("intfloat/multilingual-e5-small", device=DEVICE)

def embed_passages(texts):
    return embedder.encode(["passage: " + t for t in texts], normalize_embeddings=True,
                           batch_size=64, show_progress_bar=True).astype("float32")

def embed_query(q):
    return embedder.encode(["query: " + q], normalize_embeddings=True).astype("float32")

emb = embed_passages([c["text"] for c in chunks])
index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)
print("FAISS index size:", index.ntotal)

In [ ]:
from rank_bm25 import BM25Okapi

def tokenize(t):
    return re.findall(r"\w+", t.lower())

bm25 = BM25Okapi([tokenize(c["text"]) for c in chunks])

def dense_search(q, k=5):
    _, ids = index.search(embed_query(q), k)
    return [int(i) for i in ids[0] if i != -1]

def bm25_search(q, k=5):
    sc = bm25.get_scores(tokenize(q))
    return [int(i) for i in np.argsort(sc)[::-1][:k]]

def hybrid_search(q, k=5, pool=20, rrf_k=60):
    scores = {}
    for ranking in (dense_search(q, pool), bm25_search(q, pool)):
        for rank, i in enumerate(ranking):
            scores[i] = scores.get(i, 0) + 1 / (rrf_k + rank + 1)
    return [i for i, _ in sorted(scores.items(), key=lambda x: -x[1])][:k]

SEARCH = {"dense": dense_search, "bm25": bm25_search, "hybrid": hybrid_search}

# Quick test
for mode in SEARCH:
    ids = SEARCH[mode]("What is multi-head attention?", 3)
    print(mode, "->", [(i, "page " + str(chunks[i]["page"])) for i in ids])

## 5. LLM load karna (Answer generation)
`Qwen2.5-1.5B-Instruct` open-source hai, login ya API key ki zaroorat nahi. Pehli baar download mein 2 se 4 minute lagte hain.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(
    LLM_NAME, torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32
).to(DEVICE)
llm.eval()

def chat(messages, max_new_tokens=300):
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print("LLM ready ✅")

## 6. RAG pipeline: Retrieve → Prompt → Generate
Prompt mein LLM ko hukam hai ke jawab **sirf context se** de, aur agar jawab na mile to saaf bole. Isi se hallucination kam hoti hai.

In [ ]:
SYSTEM_PROMPT = (
    "You are a careful assistant that answers questions about a document. "
    "Use ONLY the provided context. If the answer is not in the context, reply exactly: "
    "'I could not find this in the document.' "
    "Cite the page numbers you used like (p. 3). "
    "Reply in the same language as the question."
)

def ask(question, mode="hybrid", k=4):
    ids = SEARCH[mode](question, k)
    ctx = [chunks[i] for i in ids]
    context = "\n\n".join(f"[Page {c['page']}] {c['text']}" for c in ctx)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]
    return chat(messages), ctx

answer, ctx = ask("What is multi-head attention and why is it useful?")
print("Answer:\n", answer)
print("\nSources: pages", sorted({c["page"] for c in ctx}))

In [ ]:
test_questions = [
    "How many layers does the Transformer encoder have?",
    "What optimizer was used for training?",
    "Who is the president of France?",   # document mein nahi hai: model ko mana karna chahiye
]
for q in test_questions:
    a, c = ask(q)
    print("Q:", q)
    print("A:", a, "\n" + "-" * 60)

## 7. Evaluation 📊 (Is project ka sab se strong hissa)

**Sawal:** Kaun sa retrieval method behtar hai: dense, BM25 ya hybrid?

**Tareeqa:**
1. 30 random chunks lete hain.
2. LLM se har chunk ke liye ek sawal banwate hain (jis ka jawab sirf usi chunk mein ho).
3. Har method se check karte hain ke sahi chunk top-k mein aaya ya nahi.

**Metrics:** `Hit@k` (sahi chunk top-k mein hai?) aur `MRR` (sahi chunk kitna upar aaya).

In [ ]:
def make_question(passage):
    messages = [
        {"role": "system", "content": "Write ONE specific question that can be answered using only the passage. "
                                      "Do not mention 'the passage'. Output only the question."},
        {"role": "user", "content": passage},
    ]
    return chat(messages, max_new_tokens=60).split("\n")[0].strip()

N = min(30, len(chunks))
sample = random.sample(chunks, N)
qa = []
for c in sample:
    q = make_question(c["text"])
    if len(q) > 10:
        qa.append({"question": q, "chunk_id": c["id"]})

print("Generated questions:", len(qa))
for item in qa[:5]:
    print("-", item["question"])

In [ ]:
def evaluate(mode, qa, ks=(1, 3, 5)):
    hits = {k: 0 for k in ks}
    rr = 0.0
    for item in qa:
        ranked = SEARCH[mode](item["question"], 10)
        if item["chunk_id"] in ranked:
            pos = ranked.index(item["chunk_id"]) + 1
            rr += 1 / pos
            for k in ks:
                if pos <= k:
                    hits[k] += 1
    n = len(qa)
    return {"Method": mode, **{f"Hit@{k}": hits[k] / n for k in ks}, "MRR": rr / n}

eval_df = pd.DataFrame([evaluate(m, qa) for m in SEARCH]).round(3)
eval_df

In [ ]:
eval_df.set_index("Method").plot(kind="bar", figsize=(9, 5), ylim=(0, 1.05))
plt.title("Retrieval Evaluation: Dense vs BM25 vs Hybrid")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("images/retrieval_eval.png", dpi=150)
plt.show()

**Apne observations yahan likhein:**
- Kaun sa method sab se behtar raha? Kya hybrid ne dono se behtar ya barabar result diya?
- ⚠️ **Limitation:** sawal LLM ne chunk dekh kar banaye hain, is liye un mein aksar chunk ke wohi alfaaz hote hain. Isse BM25 ko faida milta hai. Asli users ke sawal alag alfaaz mein hote hain, is liye dense retrieval asal mein shayad behtar kaam kare.

## 8. Gradio Demo 🎛️

In [ ]:
import gradio as gr

def gradio_ask(question, mode, top_k):
    if not question or not question.strip():
        return "Please type a question.", ""
    answer, ctx = ask(question, mode=mode, k=int(top_k))
    sources = "\n\n".join(f"[Page {c['page']}] {c['text'][:300]}..." for c in ctx)
    return answer, sources

demo = gr.Interface(
    fn=gradio_ask,
    inputs=[
        gr.Textbox(lines=2, label="Question"),
        gr.Dropdown(["hybrid", "dense", "bm25"], value="hybrid", label="Retrieval method"),
        gr.Slider(1, 8, value=4, step=1, label="Top-k chunks"),
    ],
    outputs=[gr.Textbox(label="Answer", lines=6), gr.Textbox(label="Retrieved sources", lines=10)],
    title="🤖 RAG PDF Chatbot",
    description="Ask questions about the loaded PDF. Answers come only from the document, with page citations.",
)
demo.launch(share=True)

## 9. Images download karna (GitHub ke liye)

In [ ]:
!zip -qr images.zip images
from google.colab import files
files.download("images.zip")

## ✅ Conclusion
- Poora RAG pipeline banaya: chunking → embeddings → FAISS + BM25 → hybrid (RRF) → LLM answer with citations.
- Retrieval ko metrics se measure kiya, sirf "demo chal gaya" par nahi ruke.
- **Limitations:** chhota LLM (1.5B) lambe ya mushkil jawabon mein ghalti kar sakta hai, Urdu generation limited hai, scanned PDFs (images) ke liye OCR chahiye, aur evaluation synthetic questions par hai.

**Future work:** cross-encoder reranker, OCR (scanned PDFs), multi-PDF support, RAGAS jaisi faithfulness evaluation, bara LLM (7B, 4-bit).